# TAC Zone Analysis

**Notebook 02 in the Curtailed-2-Compute Analysis Workflow**

This notebook converts curtailment data by TAC (Transmission Access Charge) zone using data from the CAISO transmission plan report. TAC zones are important for understanding how curtailment varies across different transmission areas and for cost allocation analysis.

## Overview

This notebook:
1. Loads CAISO curtailment data
2. Maps curtailment to TAC zones based on transmission planning data
3. Analyzes curtailment patterns by zone
4. Prepares zone-specific data for downstream analysis

## Workflow Position

- **Input**: Uses curtailment data from `01_curtailed_energy_analysis.ipynb`
- **Output**: TAC zone-mapped data used in `03_lmp_vectors.ipynb` and subsequent analyses

## Key Concepts

**TAC Zones**: Transmission Access Charge zones are used for cost allocation in the CAISO grid. While they don't affect physical power flows, analyzing curtailment by TAC zone helps:
- Identify which utility areas are absorbing or shedding excess renewables
- Understand local vs. system-wide curtailment patterns
- Support transmission planning and congestion analysis

## Data Sources

- CAISO curtailment data (from Notebook 01)
- CAISO transmission plan reports (TAC zone mappings)
- Renewable zone data from `caiso_wind_solar_pcm.csv`



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
from matplotlib.patches import Patch

## Data Requirements

### Reference Data Files (Included in Repository)

The following reference files are included in `../data/`:

- **`caiso_wind_solar_pcm.csv`**: Renewable zone curtailment projections
  - Source: CAISO transmission plan reports
  - Contains: Generation and curtailment projections by renewable zone for 2034 and 2039

- **`congestion_costs.csv`**: Transmission congestion cost projections
  - Source: CAISO transmission plan reports
  - Contains: Projected congestion costs by transmission corridor

- **`local_generation_mix_fresno.csv`**: Generation capacity mix projections
  - Source: CAISO transmission planning data
  - Contains: Projected capacity by resource type for PG&E Fresno area

- **`transmission_project_costs.csv`**: Transmission project cost data
  - Source: CAISO transmission planning documents
  - Contains: Project information and cost estimates

**Note**: These files are small reference datasets included in the repository. If you need updated data, refer to the latest CAISO transmission plan reports.



In [ ]:
from pathlib import Path
# Reference data files are in ../data/
# Fallback to root data directory if reference/ doesn't exist
data_dir = Path("../data")
if not (data_dir / "caiso_wind_solar_pcm.csv").exists():
    data_dir = Path("../data")
    print(f"Using data directory: {data_dir}")
else:
    print(f"Using reference data directory: {data_dir}")

In [ ]:
# Load renewable zone curtailment projections
try:
    pcm_file = data_dir / "caiso_wind_solar_pcm.csv"
    if not pcm_file.exists() and data_dir.name == 'reference':
        pcm_file = Path("../data") / "caiso_wind_solar_pcm.csv"
    
    pcm_df = pd.read_csv(pcm_file)
    
    # Validate data
    if pcm_df.empty:
        raise ValueError(f"Loaded {pcm_file.name} but it is empty")
    
    required_columns = ['Renewable_Zone', 'Curtailment_Ratio_Pct_2034', 'Curtailment_Ratio_Pct_2039']
    missing_cols = [col for col in required_columns if col not in pcm_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in {pcm_file.name}: {missing_cols}")
    
    print(f"[OK] Loaded {pcm_file.name}: {len(pcm_df)} zones")
    print(f"  Columns: {list(pcm_df.columns)}")
    
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find caiso_wind_solar_pcm.csv in {data_dir} or ../data/. "
        "This file should be included in the repository in data/."
    )
except Exception as e:
    raise RuntimeError(f"Error loading caiso_wind_solar_pcm.csv: {e}")

In [6]:
# Sort the DataFrame by the 2039 curtailment ratio in descending order
pcm_sorted = pcm_df.sort_values(by='Curtailment_Ratio_Pct_2034', ascending=False)

In [ ]:
colors = ['#1f77b4' if state == 'CA' else '#c7c7c7' for state in pcm_sorted['State']]
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 10))

ax.barh(pcm_sorted['Renewable_Zone'], pcm_sorted['Curtailment_Ratio_Pct_2034'], color=colors)

ax.invert_yaxis()

ax.set_xlabel('Projected Curtailment Ratio (%) in 2034', fontsize=12)
ax.set_ylabel('Renewable Zone', fontsize=12)
ax.set_title('Ranked Renewable Curtailment by Zone (2034 Forecast)', fontsize=16, pad=20)


legend_elements = [Patch(facecolor='#1f77b4', edgecolor='black', label='California Zones'),
                   Patch(facecolor='#c7c7c7', edgecolor='black', label='Out-of-State / Regional Zones')]
ax.legend(handles=legend_elements, loc='lower right')


ax.xaxis.grid(True, linestyle='--', which='major', color='grey', alpha=.25)

plt.tight_layout()

plt.show()

In [ ]:
pcm_sorted = pcm_df.sort_values(by='Curtailment_Ratio_Pct_2039', ascending=True)

y_pos = np.arange(len(pcm_sorted))
height = 0.4

colors_2034 = ['#aec7e8' if state == 'CA' else '#e0e0e0' for state in pcm_sorted['State']]
colors_2039 = ['#1f77b4' if state == 'CA' else '#999999' for state in pcm_sorted['State']]

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(14, 12))

ax.barh(y_pos + height / 2, pcm_sorted['Curtailment_Ratio_Pct_2034'], height, label='2034', color=colors_2034)
ax.barh(y_pos - height / 2, pcm_sorted['Curtailment_Ratio_Pct_2039'], height, label='2039', color=colors_2039)

ax.set_yticks(y_pos)
ax.set_yticklabels(pcm_sorted['Renewable_Zone'])

ax.set_xlabel('Projected Curtailment Ratio (%)', fontsize=12)
ax.set_ylabel('Renewable Zone', fontsize=12)
ax.set_title('Ranked Renewable Curtailment by Zone (2034 vs. 2039 Forecast)', fontsize=16, pad=20)

legend_elements = [
    Patch(facecolor='#1f77b4', edgecolor='black', label='California Zones (2039)'),
    Patch(facecolor='#aec7e8', edgecolor='black', label='California Zones (2034)'),
    Patch(facecolor='#999999', edgecolor='black', label='Out-of-State / Regional Zones (2039)'),
    Patch(facecolor='#e0e0e0', edgecolor='black', label='Out-of-State / Regional Zones (2034)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

ax.xaxis.grid(True, linestyle='--', which='major', color='grey', alpha=.25)

plt.tight_layout()
plt.show()

# Export Ranked Renewable Curtailment by Zone (2034 vs 2039)
zone_curtailment_export = pcm_sorted[['Renewable_Zone', 'State', 'Curtailment_Ratio_Pct_2034', 'Curtailment_Ratio_Pct_2039']].copy()
zone_curtailment_export.to_csv('ranked_curtailment_by_zone_2034_vs_2039.csv', index=False)
print("[OK] Exported: ranked_curtailment_by_zone_2034_vs_2039.csv")

In [ ]:
# Load congestion costs data
try:
    congestion_file = data_dir / 'congestion_costs.csv'
    if not congestion_file.exists() and data_dir.name == 'reference':
        congestion_file = Path("../data") / 'congestion_costs.csv'
    
    congestion_df = pd.read_csv(congestion_file)
    
    # Validate data
    if congestion_df.empty:
        raise ValueError(f"Loaded {congestion_file.name} but it is empty")
    
    print(f"[OK] Loaded {congestion_file.name}: {len(congestion_df)} corridors")
    
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find congestion_costs.csv in {data_dir} or ../data/. "
        "This file should be included in the repository in data/."
    )
except Exception as e:
    raise RuntimeError(f"Error loading congestion_costs.csv: {e}")

In [ ]:
congestion_df.head()

In [12]:
congestion_df = congestion_df.sort_values(by='2034_cost_mill', ascending=False)

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 8))

# Fix FutureWarning: Assign y to hue and set legend=False
sns.barplot(x='2034_cost_mill', y='constrained_area', data=congestion_df, 
            hue='constrained_area', palette='viridis', ax=ax, legend=False)

ax.set_xlabel('Projected Annual Congestion Cost in 2034 (Millions of $)', fontsize=12)
ax.set_ylabel('Transmission Corridor / Area', fontsize=12)
ax.set_title('Ranked Economic Impact of Grid Congestion (2034 Forecast)', fontsize=16, pad=20)

for container in ax.containers:
    ax.bar_label(container, fmt='$%.1fM', label_type='edge', fontsize=10, padding=3)

plt.tight_layout()
plt.show()

# Export Ranked Economic Impact of Grid Congestion (2034 Forecast)
congestion_df.to_csv('ranked_economic_impact_congestion_2034.csv', index=False)
print("[OK] Exported: ranked_economic_impact_congestion_2034.csv")

In [ ]:
# Load generation mix data
try:
    mix_file = data_dir / 'local_generation_mix_fresno.csv'
    if not mix_file.exists() and data_dir.name == 'reference':
        mix_file = Path("../data") / 'local_generation_mix_fresno.csv'
    
    mix_df = pd.read_csv(mix_file)
    
    # Validate data
    if mix_df.empty:
        raise ValueError(f"Loaded {mix_file.name} but it is empty")
    
    print(f"[OK] Loaded {mix_file.name}: {len(mix_df)} resource types")
    
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find local_generation_mix_fresno.csv in {data_dir} or ../data/. "
        "This file should be included in the repository in data/."
    )
except Exception as e:
    raise RuntimeError(f"Error loading local_generation_mix_fresno.csv: {e}")

In [ ]:
# Sort by 2039 capacity for a cleaner chart
df_mix_sorted = mix_df.sort_values(by='2039_capacity_MW', ascending=False)

plt.style.use('seaborn-v0_8-whitegrid')

fig, ax = plt.subplots(figsize=(14, 8))

x_indexes = np.arange(len(df_mix_sorted))
width = 0.4

rects1 = ax.bar(x_indexes - width/2, df_mix_sorted['2034_capacity_MW'], width, label='2034', color='#a1c9f4')
rects2 = ax.bar(x_indexes + width/2, df_mix_sorted['2039_capacity_MW'], width, label='2039', color='#1f77b4')

ax.set_ylabel('Capacity (MW)', fontsize=12)
ax.set_title('Projected Generation Capacity Mix in PG&E Fresno Area (2034 vs. 2039)', fontsize=16, pad=20)
ax.set_xticks(x_indexes)
ax.set_xticklabels(df_mix_sorted['resource_type'], rotation=30, ha='right', fontsize=11)
ax.legend()

ax.bar_label(rects1, padding=3, fmt='%d', fontsize=9)
ax.bar_label(rects2, padding=3, fmt='%d', fontsize=9)

ax.yaxis.grid(True, linestyle='--', which='major', color='grey', alpha=.25)

fig.tight_layout()
plt.show()

# Export Projected Generation Capacity Mix in PG&E Fresno Area
df_mix_sorted.to_csv('generation_capacity_mix_fresno_2034_vs_2039.csv', index=False)
print("[OK] Exported: generation_capacity_mix_fresno_2034_vs_2039.csv")
